### Step 1 — Imports & Configuration

Import all required libraries and set global constants used throughout the pipeline.

> **Note:** Run this cell first every time you open the notebook. All variables defined here are used by subsequent cells.

In [1]:
import re
import os
import time
import shutil
import torch
import pytesseract
from PIL import Image
from langchain_core.documents import Document
from  langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from langchain_community.document_loaders import  PyMuPDFLoader, Docx2txtLoader, TextLoader,  BSHTMLLoader


os.environ["USER_AGENT"] = "Mozilla/5.0 (RAG-Telecom-Bot)"


CHROMA_DB_PATH  = "./chroma_telecom_db"
COLLECTION_NAME = "telecom_egypt"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 150
TOP_K           = 5


if os.path.exists("./chroma_telecom_db"):
    shutil.rmtree("./chroma_telecom_db")
    print(" Old DB deleted")

 Old DB deleted


### Step 2 — Knowledge Base URLs

Define the Telecom Egypt pages to scrape, organized by category.
Each category is stored as metadata with every document — this enables filtered retrieval later.

| Category | Description |
|---|---|
| mobile | Prepaid, postpaid, and internet mobile plans |
| home_internet | WE Internet, Landline, WE Air plans |
| promotions | Current offers and discounts |
| support | Help & FAQ pages |
| business | Business plans and connectivity |
| about | Company info and contact details |

In [2]:
TELECOM_URLS = {
    "mobile": [
        "https://te.eg/wps/portal/te/Personal/Mobile/Prepaid-12PT",
        "https://te.eg/wps/portal/te/Personal/Mobile/Control-WE-MIX",
        "https://te.eg/wps/portal/te/Personal/Mobile/WE-Gold",
        "https://te.eg/wps/portal/te/Personal/Mobile/Nitro-mobile-internet",
        "https://te.eg/wps/portal/te/Personal/Mobile/Nitro-mifi/",
    ],
    "home_internet": [
        "https://te.eg/wps/portal/te/Personal/WEInternet/",
        "https://te.eg/wps/portal/te/Personal/WELandline/",
        "https://te.eg/wps/portal/te/Personal/WE-Air-Prepaid",
        "https://te.eg/wps/portal/te/Personal/WE-Air-Postpaid",
    ],
    "promotions": [
        "https://te.eg/wps/portal/te/Personal/Promotions/Internet-Promotions/",
        "https://te.eg/wps/portal/te/Personal/Promotions/Mobile-Promotions/",
        "https://te.eg/wps/portal/te/Personal/Promotions/Landline-Promotions/",
        "https://te.eg/wps/portal/te/Personal/All-Promotions/",
    ],
    "support": [
        "https://te.eg/wps/portal/te/Personal/Help And Support",
    ],
    "business": [
        "https://te.eg/wps/portal/te/Business/Mobile-Services/WE-Business",
        "https://te.eg/wps/portal/te/Business/Mobile-Services/WE-Business-Value",
        "https://te.eg/wps/portal/te/Business/Mobile-Services/WE-Business-Internet",
        "https://te.eg/wps/portal/te/Business/Data-Connectivity/Business-ADSL",
        "https://te.eg/wps/portal/te/Business/Data-Connectivity/IP-VPN",
    ],
    "about": [
        "https://www.te.eg/wps/portal/te/About/About Us/",
        "https://www.te.eg/wps/portal/te/About/Contact Us/",
    ]
}

### Step 3 — Text Cleaning

Remove navigation menus and footer noise that Selenium picks up alongside the main content.

- `start_markers` — cuts everything before the actual page content begins
- `footer_markers` — cuts everything after the page content ends

In [3]:
def clean_text(text) -> str:
    start_markers = [
        "Investor Relations CSR",
    ]
    for marker in start_markers:
        if marker in text:
            text = text[text.find(marker) + len(marker):]

    footer_markers = [
        "For all mobile, internet and fixed line users",
        "TELECOMEGYPT",
        "Follow Us",
        "Download Our App",
        "Copyright",
    ]
    for marker in footer_markers:
        if marker in text:
            text = text[:text.find(marker)]

    text = re.sub(r'\s+', ' ', text).strip()
    return text

### Step 4 — Web Scraping with Selenium

Scrape all URLs using a headless Chrome browser.

We use Selenium instead of a simple HTTP request because te.eg renders its content dynamically with JavaScript — a static request would return an empty page.


In [4]:
def load_telecom_docs(url_dict: dict) :
    all_docs = []

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--lang=ar")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    for category, urls in url_dict.items():
        for url in urls:
            try:
                driver.get(url)
                time.sleep(3)

                text = driver.find_element("tag name", "body").text
                text = clean_text(text)

                if len(text) > 100:
                    doc = Document(
                        page_content=text,
                        metadata={
                            "source": url,
                            "category": category,
                            "source_site": "te.eg"
                        }
                    )
                    all_docs.append(doc)
                    print(f"OK {category}: {url.split('/')[-1]}")
                else:
                    print(f"Empty: {url.split('/')[-1]}")

            except Exception as e:
                print(f"Failed: {url} - {e}")

    driver.quit()
    print(f"\nTotal: {len(all_docs)} documents")
    return all_docs



raw_docs = load_telecom_docs(TELECOM_URLS)

OK mobile: Prepaid-12PT
OK mobile: Control-WE-MIX
OK mobile: WE-Gold
OK mobile: Nitro-mobile-internet
OK mobile: 
OK home_internet: 
OK home_internet: 
OK home_internet: WE-Air-Prepaid
OK home_internet: WE-Air-Postpaid
OK promotions: 
OK promotions: 
OK promotions: 
OK promotions: 
OK support: Help And Support
OK business: WE-Business
OK business: WE-Business-Value
OK business: WE-Business-Internet
OK business: Business-ADSL
OK business: IP-VPN
OK about: 
OK about: 

Total: 21 documents


### Step 5 — Chunking

Split documents into smaller chunks for embedding.


- `chunk_size=800` — large enough to contain meaningful context, small enough for accurate retrieval
- `chunk_overlap=150` — prevents information loss at chunk boundaries
- Arabic-aware separators ensure sentences are not cut mid-way



In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
        separators=[
        "\n\n",  # paragraph
        "\n",    # new line
        ".",     # English period
        ".",     # Arabic period
        "،",     # Arabic comma
        " ",     # space
        "",      # last resort
    ],
    length_function=len,)

chunks = text_splitter.split_documents(raw_docs)

print(f"Raw Documnts: {len(raw_docs)}")
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks)// len(chunks)} characters")


Raw Documnts: 21
Total chunks: 53
Avg chunk size: 573 characters


### Step 6 — Multilingual Embeddings

Convert each chunk into a numerical vector using a multilingual model.

We use `intfloat/multilingual-e5-large` because:
- Supports 50+ languages including Arabic and Egyptian dialect
- Understands that "إنترنت", "internet", and "نت" have the same meaning
- Runs fully offline — no API calls needed



In [6]:

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

test_vector = embeddings.embed_query("ما هي باقات الإنترنت المتاحة؟")
print(f" Embedding model loaded")
print(f" Vector dimensions: {len(test_vector)}")

C:\Users\mohamed.mostafa\AppData\Local\Temp\ipykernel_27172\1366425528.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embedding model loaded
 Vector dimensions: 1024


### Step 7 — Store in ChromaDB

Save all vectors to a persistent local database.

ChromaDB stores everything on disk at `./chroma_telecom_db`.
On subsequent runs, load it directly without re-indexing:




In [7]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="telecom_egypt",
    persist_directory="./chroma_telecom_db",
)

print(f" Vector store created")
print(f"Total vectors stored: {vector_store._collection.count()}")

 Vector store created
Total vectors stored: 53


### Step 8 — Retriever

Search the vector store for the most relevant chunks at query time.

We use **MMR (Maximal Marginal Relevance)** instead of plain similarity search because MMR returns results that are both relevant AND diverse — it avoids returning 5 chunks that all say the same thing.

- `k=5` — return top 5 chunks
- `fetch_k=20` — consider top 20 candidates before MMR selection
- `lambda_mult=0.7` — balance between relevance (1.0) and diversity (0.0)

In [8]:

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5 , "fetch_k": 20, "lambda_mult": 0.7}
)


query = "ما هي باقات الإنترنت المتاحة في البيت؟"
results = retriever.invoke(query)

print(results)

[Document(id='ad84e6b4-a2ff-4d8d-96ba-86fd5c6aaaca', metadata={'source_site': 'te.eg', 'category': 'mobile', 'source': 'https://te.eg/wps/portal/te/Personal/Mobile/WE-Gold'}, page_content='. يمكنك التحويل لباقة إنترنت منزلي أعلى و دفع الفرق مع الفاتورة الشهرية. يمكن احتساب مصاريف التحويل عن طريق “Fixed Internet Calculator” أو التفاصيل كما هو موضح بالأسفل: باقات الإنترنت الأرضى WE Gold و مصاريف تغير باقة الإنترنت الأرضى (ج.م) 260 525 775 1050 1300 2000 140 جيجا سوبر مجاناً مجاناً - 200 جيجا سوبر 65 65 - 250 جيجا سوبر 135 135 مجاناً مجاناً - 400 جيجا سوبر 317 317 182 182 مجاناً - 600 جيجا سوبر 494 494 359 359 177 مجاناً 1 تيرا سوبر - 790 613 أيضًا مع WE Gold يمكنك الاستمتاع بإشتراك WE Air يمكنك الإختيار ما بين باقات الإنترنت المنزلي أو باقات WE Air وليس كليهما. يتم تطبيق المصروفات الحالية ل WE Air. لتغيير باقة الإنترنت المنزلي إلى WE Air أو العكس، يجب زيارة أي من فروع WE بشرط عدم وجود أي مستحقات أو أقساط خاصة بجهاز الراوتر. للإشتراك في WE Air سوف يتم دفع رسوم إصدار الشريحة'), Document(id

### Step 9 — LLM & RAG Chain

Initialize the language model and build the full RAG chain.

**Model:** `gemma4:e2b` via Ollama — runs fully offline on your machine.

**RAG Chain flow:**
- question → retriever → format_context → prompt → LLM → answer

**The system prompt instructs the model to:**
1. Detect the user's language automatically
2. Reply in the same language (Arabic / English / Egyptian dialect)
3. Use only the retrieved context — no hallucination
4. Include source URLs at the end of every answer

In [9]:
llm = ChatOllama(
    model="gemma4:e2b",
   
)

# SYSTEM_PROMPT = """You are a Telecom Egypt AI assistant.

# *CRITICAL LANGUAGE RULE (STRICT):
# 1. DETECT the language of the user's message.
# 2. RESPOND IN THAT EXACT LANGUAGE.
# 3. If User speaks English → You MUST reply in English (Translate facts from context).
# 4. If User speaks Arabic → You MUST reply in Egyptian Arabic.
# 5. Never switch language.

# * Tone: Friendly, helpful, concise.
# * Sources: Always list source URLs at the end.

# Examples:
# User: How do I pay my bill?
# Assistant: You can pay online via the WE website or app.

# User: النت فاصل عندي اعمل ايه؟
# Assistant: بص، جرب الأول تتأكد من الوصلات عندك، وبعدين جرب تعمل restart للراوتر. لو المشكلة لسه موجودة، كلم 111 وهيساعدوك.

# Context:
# {context}
# """

# Ensure your LLM definition uses the correct model:





SYSTEM_PROMPT = """
You are a Telecom Egypt (WE) AI assistant.

 CRITICAL LANGUAGE RULES (STRICT):
1. ALWAYS detect the language/dialect of the USER'S LATEST MESSAGE ONLY.
2. RESPOND IN THAT EXACT LANGUAGE. DO NOT continue the language from previous messages.
3. If English → Must Reply in English
4. If Egyptian → Must Reply in friendly Egyptian dialect (عامية مصرية)
5. Mixed/LANGUAGE -switching → Mirror their exact style naturally

 Tone: Friendly, helpful, concise (like a real WE support agent).
 Grounding: Answer ONLY using the provided context. Never hallucinate.
 Sources: Always list source URLs at the end.

Examples:

User: how can I check my bill?
Assistant: You can check your bill from the WE website or app.

User: النت فاصل عندي اعمل ايه؟
Assistant: بص، جرب تأكد من الوصلات، وبعدين جرب تعمل restart للراوتر. لو المشكلة لسه موجودة كلم 111 وهيساعدوك.


Context:
{context}
"""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

def format_context(docs: list) -> str:
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "te.eg")
        parts.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

rag_chain = (
    {
        "context":  retriever | format_context,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("ما هي خطط الإنترنت المتاحة في المنزل ؟")
print(answer)

ممكن تحسب سعة الإنترنت الأرضي حسب احتياجك وتشوف الخيارات المتاحة ليك عشان تحصل على إنترنت أرضي أعلى. وكمان تقدر تحسب المصاريف الإضافية.

ممكن تحول لباقة إنترنت منزلي بسعة أكبر لتناسب احتياجاتك.

(المعلومات متوفرة في المصادر المرفقة)
[Source: https://te.eg/wps/portal/te/Personal/Mobile/WE-Gold]
[Source: https://te.eg/wps/portal/te/Personal/Mobile/WE-Gold]
[Source: https://te.eg/wps/portal/te/Personal/Promotions/Internet-Promotions/]


### Step 10 — User Document Upload

Allow users to upload their own documents and query them alongside te.eg content.

**Supported formats:**

| Format | Loader |
|---|---|
| PDF | PyMuPDFLoader |
| DOCX | Docx2txtLoader |
| TXT | TextLoader |
| HTML | BSHTMLLoader |
| Images (JPG, PNG) | Tesseract OCR |

Uploaded documents are chunked and added to the same ChromaDB collection — the chatbot searches both te.eg content and user documents in every query.

In [ ]:
def load_uploaded_file(file_path: str) :

    pytesseract.pytesseract.tesseract_cmd = r"RAG_Telecom_Egypt\\Tesseract-OCR\\tesseract.exe"

    
    ext = file_path.split(".")[-1].lower()
    docs = []
    
    try:
        if ext == "pdf":
            loader = PyMuPDFLoader(file_path)
            docs = loader.load()
            
        elif ext == "docx":
            loader = Docx2txtLoader(file_path)
            docs = loader.load()
            
        elif ext == "txt":
            loader = TextLoader(file_path, encoding="utf-8")
            docs = loader.load()
            
        elif ext in ["html"]:
            loader = BSHTMLLoader(file_path)
            docs = loader.load()
            
        elif ext in ["jpg", "jpeg", "png"]:
        
            img = Image.open(file_path)
            text = pytesseract.image_to_string(img, lang="ara+eng")
            if text.strip():
                docs = [Document(
                    page_content=text,
                    metadata={"source": file_path, "type": "image"}
                )]
            else:
                print(f" No text found in image: {file_path}")
                
        else:
            print(f" Unsupported file type: .{ext}")
            return []
        

     
        for doc in docs:
            doc.metadata["uploaded"] = True
            doc.metadata["file_type"] = ext
            doc.metadata["filename"] = os.path.basename(file_path)

        
        print(f" Loaded {len(docs)} pages from {os.path.basename(file_path)}")
        return docs
        
    except Exception as e:
        print(f" Error loading {file_path}: {e}")
        return []
    
def process_uploaded_files(docs):
    processed_docs = []

    for doc in docs:
        doc.page_content = doc.page_content.strip()

        doc.metadata["category"] = "user_upload"

        processed_docs.append(doc)

    return processed_docs  




def add_file_to_store(file_path):
    docs = load_uploaded_file(file_path)

    if not docs:
        return 0

    docs = process_uploaded_files(docs)

    chunks = text_splitter.split_documents(docs)

    vector_store.add_documents(chunks)

    print(f"Added {len(chunks)} chunks from {os.path.basename(file_path)}")
    return len(chunks)

### Step 11 — Gradio Chat Interface

Build the chat UI with WE brand colors and multilingual example questions.

**Features:**
- Chat history with memory (last 5 turns)
- File upload for documents and images
- Source citations displayed with every answer
- WE brand colors (#003399 blue, #FFD700 gold)
- Clear chat button to reset conversation memory

Run the next cell to launch the interface, then open: `http://127.0.0.1:7860`

In [11]:
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory

from langchain_classic.chains import ConversationalRetrievalChain



memory = ConversationBufferMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": prompt},
)

def reset_memory():
    memory.chat_memory.clear()
    

def chat(message: str, history: list, uploaded_file) -> str:
    if uploaded_file is not None:
        add_file_to_store(uploaded_file.name)

    result  = conv_chain.invoke({"question": message})
    answer  = result["answer"]
    sources = result.get("source_documents", [])

    if sources:
        unique_sources = list(set(
            doc.metadata.get("source", "")
            for doc in sources
            if doc.metadata.get("source_site") == "te.eg"
        ))
        if unique_sources:
            answer += "\n\nSources:"
            for src in unique_sources:
                answer += f"\n- {src}"

    return answer

css = """
.gradio-container { background-color: #FFFFFF; }
.chatbot .message.bot { background-color: #F0F4FF; border-right: 4px solid #003399; }
.chatbot .message.user { background-color: #003399; color: white; }
footer { display: none; }
"""



with gr.Blocks(title="Telecom Egypt Intelligent Assistant", theme=gr.themes.Soft( primary_hue= gr.themes.colors.purple), css = css) as demo :
    gr.Markdown("""
    <div style='background-color:#003399; padding:20px; border-radius:8px; margin-bottom:10px'>
        <h1 style='color:white; margin:0'>Telecom Egypt — WE Assistant</h1>
        <p style='color:#FFD700; margin:5px 0 0'>مساعدك الذكي  WE</p>
    </div>
    """)

    clear_btn = gr.Button("Clear Chat")
    clear_btn.click(fn=reset_memory, outputs=[])


    gr.ChatInterface(
        fn=chat,
        additional_inputs=[
            gr.File(
                label="Upload Document (PDF, DOCX, TXT, Image)",
                file_types=[".pdf", ".docx", ".txt", ".html", ".png", ".jpg", ".jpeg"],
            )
        ],
        examples=[
            ["ما هي خطط الإنترنت المتاحة في المنزل؟"],
            ["What are the prepaid mobile plans?"],
            ["في عروض على النت دلوقتي؟"],
            ["عايز أعرف أسعار خطط WE Gold"],
            ["How do I pay my bill online?"],
        ],
    )



C:\Users\mohamed.mostafa\AppData\Local\Temp\ipykernel_27172\1574521264.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(
C:\Users\mohamed.mostafa\AppData\Local\Temp\ipykernel_27172\1574521264.py:57: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Telecom Egypt Intelligent Assistant", theme=gr.themes.Soft( primary_hue= gr.themes.colors.purple), css = css) as demo :


### Step 12 — Launch

Start the Gradio app. Set `share=True` to get a public temporary URL valid for 72 hours.

Alternatively, run the standalone app from terminal:
```bash
python app.py
```

In [12]:
reset_memory()

In [13]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d72eafc063682a1efb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
